## 1. Set up your environment

This step ensures all the necessary tools are available to build the RAG system. Each library serves a specific role: Langchain handles the orchestration of components, transformers provide pre-trained models, sentence-transformers generate embeddings, datasets load sample data, and FAISS enables fast similarity searches.

In [ ]:
!pip install -q langchain
!pip install -q torch
!pip install -q transformers
!pip install -q sentence-transformers
!pip install -q datasets
!pip install -q faiss-cpu
!pip install -U langchain-community


## 2. Load the dataset

To provide the system with information to retrieve from, you’ll load a real-world dataset. HuggingFaceDatasetLoader simplifies the process of accessing Hugging Face datasets and formatting them into documents that Langchain can process.

In [ ]:
# Ensure datasets library is up-to-date
!pip install -Uq datasets

from langchain.document_loaders import HuggingFaceDatasetLoader

# Specify the dataset name and content column
dataset_name = "databricks/databricks-dolly-15k"
page_content_column = "context"

# Create a HuggingFaceDatasetLoader instance and load the data as documents
loader = HuggingFaceDatasetLoader(dataset_name, page_content_column)
data = loader.load()
print(data[:2]) # Optional: Print the first 2 entries to verify loading


## 3. Split the documents

Language models have a limit on how much text they can process at once. Splitting large documents into smaller, overlapping chunks ensures that no important context is lost and that each piece of text is a manageable size for embedding and retrieval.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Create a RecursiveCharacterTextSplitter instance
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=150)

# Split the loaded documents
docs = text_splitter.split_documents(data)
print(docs[0]) # Optional: Print the first document chunk


## 4. Embed the text

Text needs to be converted into numerical representations (embeddings) so that similar pieces of text can be found efficiently. Using a sentence-transformer model creates embeddings that capture semantic meaning, enabling effective retrieval later.

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

# Define the model path, model configurations, and encoding options
modelPath = "sentence-transformers/all-MiniLM-l6-v2"
model_kwargs = {'device':'cpu'}
encode_kwargs = {'normalize_embeddings': False}

# Initialize HuggingFaceEmbeddings
embeddings = HuggingFaceEmbeddings(
  model_name=modelPath,
  model_kwargs=model_kwargs,
  encode_kwargs=encode_kwargs
)

# (Optional) Test embedding creation
text = "This is a test document."
query_result = embeddings.embed_query(text)
print(query_result[:3])


## 5. Create a vector store

A vector store like FAISS indexes the embeddings, allowing fast and scalable similarity searches. This is how the system quickly finds relevant pieces of text when a query is made.

In [ ]:
from langchain_community.vectorstores import FAISS

# Create a FAISS vector store from the document chunks and embeddings
db = FAISS.from_documents(docs, embeddings)


## 6. Prepare the LLM model

The Language Model is responsible for generating answers based on retrieved documents. Loading a pre-trained model and wrapping it in a Langchain pipeline makes it easy to integrate with the retrieval system.

In [ ]:
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, pipeline
from langchain_community.llms import HuggingFacePipeline

# Load the tokenizer and question-answering model
model_name = "Intel/dynamic_tinybert"
tokenizer = AutoTokenizer.from_pretrained(model_name, padding=True, truncation=True, max_length=512)
model = AutoModelForQuestionAnswering.from_pretrained(model_name)

# Create a question-answering pipeline
Youtubeer = pipeline(
  "question-answering",
  model=model,
  tokenizer=tokenizer,
  return_tensors='pt'
)

# Create a Langchain pipeline wrapper
llm = HuggingFacePipeline(
  pipeline=Youtubeer,
  model_kwargs={"temperature": 0.7, "max_length": 512},
)


## 7. Build the Retrieval QA Chain

The Retrieval QA Chain connects the retriever (which finds relevant documents) with the LLM (which generates answers). This chain enables the full RAG process, where the system retrieves helpful context and then answers the user’s query based on that context.

In [ ]:
from langchain.chains import RetrievalQA

# Create a retriever from your FAISS database
retriever = db.as_retriever(search_kwargs={"k": 4}) # Optional: You can adjust k for number of documents retrieved

# Build the RetrievalQA chain
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="refine", retriever=retriever, return_source_documents=False)


## 8. Test your RAG system

Running a test query allows you to verify that all components are working together. This step ensures that documents are retrieved correctly and that the model generates meaningful answers based on the retrieved context.

In [ ]:
# Define your question
question = "What is cheesemaking?"

# Run the QA chain and print the result
result = qa.run({"query": question})
print(result)
